In [10]:
!pip install -q -U gradio transformers accelerate evaluate rouge_score sentencepiece

In [11]:
import torch
import transformers
import gradio as gr

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gr.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

SYSTEM INFORMATION
PyTorch: 2.11.0+cu128
Transformers: 5.13.1
Gradio: 6.24.0
CUDA available: True
GPU: Tesla T4


In [12]:
# ============================================================
# LOAD BART SUMMARIZATION MODEL
# ============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

print("=" * 60)
print("LOADING SUMMARIZATION MODEL")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Please wait while the model downloads...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

if torch.cuda.is_available():
    model = model.to("cuda")
    device = "cuda"
else:
    device = "cpu"

model.eval()

print("\nModel loaded successfully!")
print("Device:", device)

LOADING SUMMARIZATION MODEL
Model: facebook/bart-large-cnn
Please wait while the model downloads...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


Model loaded successfully!
Device: cuda


In [13]:
# ============================================================
# SUMMARIZATION FUNCTION
# ============================================================

def summarize_text(input_text):

    if input_text is None or input_text.strip() == "":
        return "Please enter some text."

    words = input_text.split()

    if len(words) < 20:
        return "Please enter at least 20 words."

    # Tokenize input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # Move input to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate summary
    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_length=45,
            min_length=15,
            num_beams=4,
            do_sample=False,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    # Convert tokens back to text
    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary


print("Summarization function created successfully.")

Summarization function created successfully.


In [14]:
# ============================================================
# TEST BART
# ============================================================

sample_text = """
Artificial intelligence is transforming many industries around
the world. Generative AI systems can create text, images, audio,
video, and computer programs. These technologies are being used
in education, healthcare, marketing, software development,
scientific research, and customer service. Organizations are
adopting generative AI because it can automate repetitive tasks
and help people produce content more efficiently. However,
responsible development is important because AI systems can
produce inaccurate information and may create privacy, security,
bias, and copyright concerns.
"""

print("=" * 60)
print("TEST INPUT")
print("=" * 60)

print(sample_text)

print("\n" + "=" * 60)
print("GENERATED SUMMARY")
print("=" * 60)

test_summary = summarize_text(sample_text)

print(test_summary)

TEST INPUT

Artificial intelligence is transforming many industries around
the world. Generative AI systems can create text, images, audio,
video, and computer programs. These technologies are being used
in education, healthcare, marketing, software development,
scientific research, and customer service. Organizations are
adopting generative AI because it can automate repetitive tasks
and help people produce content more efficiently. However,
responsible development is important because AI systems can
produce inaccurate information and may create privacy, security,
bias, and copyright concerns.


GENERATED SUMMARY
Artificial intelligence is transforming many industries around the world. Generative AI systems can create text, images, audio, video, and computer programs. These technologies are being used in education, healthcare, marketing, software


In [15]:
# ============================================================
# CREATE GRADIO WEB APPLICATION
# ============================================================

import gradio as gr

demo = gr.Interface(
    fn=summarize_text,

    inputs=gr.Textbox(
        lines=12,
        label="Enter Text to Summarize",
        placeholder="Paste a long article here..."
    ),

    outputs=gr.Textbox(
        lines=6,
        label="Generated Summary"
    ),

    title="GenAI Text Summarizer",

    description=(
        "A Generative AI text summarization application "
        "using BART-large-CNN and Gradio."
    ),

    examples=[
        [
            """
            Renewable energy is becoming increasingly important
            as countries search for cleaner alternatives to fossil
            fuels. Solar power, wind power, hydropower and geothermal
            energy can provide electricity while reducing dependence
            on coal, oil and natural gas. Advances in technology are
            making renewable energy more affordable and accessible.
            Governments and businesses are investing in renewable
            infrastructure to reduce emissions and create a more
            sustainable energy future.
            """
        ],
        [
            """
            Cloud computing provides users with access to computing
            resources over the internet. These resources include
            servers, databases, storage, networking and software.
            Instead of purchasing large amounts of physical hardware,
            organizations can use cloud services and increase or
            decrease their resources according to demand. Cloud
            computing is widely used for application development,
            data analytics, artificial intelligence and backup.
            """
        ]
    ]
)

print("Gradio application created successfully.")

Gradio application created successfully.


In [16]:
# ============================================================
# LAUNCH GRADIO APP
# ============================================================

print("=" * 60)
print("STARTING GRADIO APPLICATION")
print("=" * 60)

demo.launch(
    share=True
)

STARTING GRADIO APPLICATION
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7b261bcc79b61772a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
# ============================================================
# LOAD ROUGE EVALUATION
# ============================================================

import evaluate

print("Loading ROUGE...")

rouge = evaluate.load("rouge")

print("ROUGE loaded successfully!")

Loading ROUGE...


ROUGE loaded successfully!


In [18]:
# ============================================================
# ROUGE EVALUATION - SINGLE EXAMPLE
# ============================================================

evaluation_text = """
Generative artificial intelligence is a rapidly developing
technology that allows computers to generate new content such
as text, images, audio, video, and software code. Large AI models
learn patterns from enormous datasets and can then produce useful
outputs based on user instructions. Generative AI is being used
in education, healthcare, marketing, customer service, software
development and scientific research. Although these systems
provide significant benefits, organizations must consider
accuracy, privacy, security, bias and responsible use when
deploying them.
"""

reference_summary = """
Generative AI can create text, images, audio, video and software
code and is increasingly used across many industries. Despite
its benefits, responsible deployment requires attention to
accuracy, privacy, security and bias.
"""

# Generate summary
generated_summary = summarize_text(evaluation_text)

print("=" * 60)
print("GENERATED SUMMARY")
print("=" * 60)

print(generated_summary)

print("\n" + "=" * 60)
print("REFERENCE SUMMARY")
print("=" * 60)

print(reference_summary)


# Calculate ROUGE
scores = rouge.compute(
    predictions=[generated_summary],
    references=[reference_summary]
)

print("\n" + "=" * 60)
print("ROUGE EVALUATION SCORES")
print("=" * 60)

for metric, score in scores.items():
    print(f"{metric}: {score:.4f}")

GENERATED SUMMARY
Generative artificial intelligence is a rapidly developing technology that allows computers to generate new content. Large AI models learn patterns from enormous datasets and can then produce useful outputs based on user instructions. Generative AI is being

REFERENCE SUMMARY

Generative AI can create text, images, audio, video and software
code and is increasingly used across many industries. Despite
its benefits, responsible deployment requires attention to
accuracy, privacy, security and bias.


ROUGE EVALUATION SCORES
rouge1: 0.1765
rouge2: 0.0303
rougeL: 0.1176
rougeLsum: 0.1765


In [19]:
# ============================================================
# MULTIPLE-SAMPLE ROUGE EVALUATION
# ============================================================

evaluation_data = [

    {
        "article": """
        Renewable energy comes from natural resources such as
        sunlight, wind, water and geothermal heat. Solar and wind
        energy are becoming increasingly popular because they can
        generate electricity with low operational carbon emissions.
        Renewable energy can reduce dependence on fossil fuels and
        support a cleaner and more sustainable energy system.
        """,

        "reference": """
        Renewable energy sources such as solar and wind can reduce
        dependence on fossil fuels and support a cleaner and more
        sustainable energy system.
        """
    },

    {
        "article": """
        Machine learning is a branch of artificial intelligence
        that enables computers to learn patterns from data. Rather
        than being explicitly programmed for every situation,
        machine learning systems use training data to learn
        relationships and make predictions. Machine learning is
        widely used in recommendation systems, fraud detection,
        speech recognition, medical analysis and image processing.
        """,

        "reference": """
        Machine learning enables computers to learn patterns from
        data and is used for predictions, recommendations, fraud
        detection, speech recognition and image processing.
        """
    },

    {
        "article": """
        Cloud computing provides access to computing resources
        through the internet. These resources include servers,
        storage, databases, networking and software. Organizations
        can increase or decrease resources according to demand
        without maintaining all the physical infrastructure
        themselves. Cloud computing is widely used for software
        development, data storage, analytics and artificial
        intelligence applications.
        """,

        "reference": """
        Cloud computing provides internet-based access to computing
        resources and allows organizations to scale resources
        according to demand.
        """
    }
]


generated_summaries = []
reference_summaries = []

print("=" * 60)
print("GENERATING SUMMARIES FOR EVALUATION")
print("=" * 60)

for i, item in enumerate(evaluation_data):

    generated = summarize_text(item["article"])

    generated_summaries.append(generated)
    reference_summaries.append(item["reference"])

    print(f"\nExample {i + 1}")
    print("-" * 60)
    print("Generated:")
    print(generated)

# Calculate ROUGE
scores = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

print("\n" + "=" * 60)
print("FINAL ROUGE RESULTS")
print("=" * 60)

for metric, score in scores.items():
    print(f"{metric}: {score:.4f}")

GENERATING SUMMARIES FOR EVALUATION

Example 1
------------------------------------------------------------
Generated:
Renewable energy comes from natural resources such as sunlight, wind, water and geothermal heat. Renewable energy can reduce dependence on fossil fuels and support a cleaner and more sustainable energy system.

Example 2
------------------------------------------------------------
Generated:
Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data. Rather than being explicitly programmed for every situation, machine learning systems use training data to learn relationships and make predictions. Machine learning is

Example 3
------------------------------------------------------------
Generated:
Cloud computing provides access to computing resources through the internet. These resources include servers, databases, networking and software. Organizations can increase or decrease resources according to demand.

FINAL ROUGE

In [20]:
# ============================================================
# SAVE RESULTS
# ============================================================

result_file = "/content/genai_summarization_results.txt"

with open(result_file, "w", encoding="utf-8") as f:

    f.write(
        "DEPLOYMENT AND EVALUATION OF A GENERATIVE AI APPLICATION\n"
    )
    f.write(
        "USING CLOUD-BASED APIs AND AI FRAMEWORKS\n"
    )
    f.write("=" * 70 + "\n\n")

    f.write("MODEL: facebook/bart-large-cnn\n")
    f.write("FRAMEWORK: Gradio\n")
    f.write("EVALUATION: ROUGE\n\n")

    for i in range(len(generated_summaries)):

        f.write(f"Example {i + 1}\n")
        f.write("-" * 50 + "\n")

        f.write("Generated Summary:\n")
        f.write(generated_summaries[i] + "\n\n")

        f.write("Reference Summary:\n")
        f.write(reference_summaries[i] + "\n\n")

    f.write("ROUGE SCORES\n")
    f.write("-" * 50 + "\n")

    for metric, score in scores.items():
        f.write(f"{metric}: {score:.4f}\n")


print("Results saved successfully:")
print(result_file)

Results saved successfully:
/content/genai_summarization_results.txt
